# 4IB — SGDRegressor (Huber Loss)

Incremental linear regression using Huber loss: residuals inside the epsilon threshold are penalized quadratically, while larger residuals receive a linear penalty. The loss is fixed; Optuna tunes epsilon and the remaining optimizer/regularization settings.

This notebook is CPU-only. It uses `partial_fit()` for epoch-wise incremental updates, Optuna TPE for hyperparameter search, and expanding-window validation on training weeks only. The final model's weights remain frozen throughout the test forecast.

**Reproducibility:** every estimator and the Optuna sampler use random state 34. The model-specific loss/architecture is fixed by this notebook; the remaining hyperparameters are tuned.

## 0. Load Stream I features and set the compute budget

In [ ]:
from pathlib import Path
import os, json, warnings, gc, time, copy, random, sys
import numpy as np
import pandas as pd

SEED = 34
random.seed(SEED)
np.random.seed(SEED)

ON_KAGGLE = Path('/kaggle/working').exists()
PROJECT_ROOT = Path('/kaggle/working/us_accidents_weekly_forecasting') if ON_KAGGLE else Path('./us_accidents_weekly_forecasting')
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
MODEL_DIR = PROJECT_ROOT / 'models'
REPORT_DIR = PROJECT_ROOT / 'reports'
for _p in (PROJECT_ROOT, ARTIFACT_DIR, MODEL_DIR, REPORT_DIR):
    _p.mkdir(parents=True, exist_ok=True)

# Set this only when more than one prior notebook output is attached as a Kaggle input.
ARTIFACT_INPUT_ROOT = None  # e.g. Path('/kaggle/input/us-accidents-weekly-stage-2')

def resolve_artifact(filename, required=True):
    candidates = []
    if ARTIFACT_INPUT_ROOT is not None:
        p = Path(ARTIFACT_INPUT_ROOT) / filename
        if p.exists():
            candidates.append(p)
    p = ARTIFACT_DIR / filename
    if p.exists():
        candidates.append(p)
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates.extend(kaggle_input.rglob(filename))
    # Deduplicate resolved paths and prefer the most recently modified attached artifact.
    unique = {str(p.resolve()): p for p in candidates}
    candidates = list(unique.values())
    if not candidates:
        if required:
            raise FileNotFoundError(
                f'Could not locate {filename}. Add the preceding notebook output as a Kaggle input, '
                f'or set ARTIFACT_INPUT_ROOT. Local output searched: {ARTIFACT_DIR}'
            )
        return None
    chosen = max(candidates, key=lambda p: p.stat().st_mtime)
    if len(candidates) > 1:
        print(f'Found {len(candidates)} copies of {filename}; using newest: {chosen}')
    return chosen

print('Project root:', PROJECT_ROOT.resolve())
print('Seed:', SEED)

import matplotlib.pyplot as plt
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import SGDRegressor
import joblib

X_train_raw = np.load(resolve_artifact('stream_i_X_train_raw.npy')).astype(np.float64)
y_train_raw = np.load(resolve_artifact('stream_i_y_train_raw.npy')).astype(np.float64)
X_test_raw = np.load(resolve_artifact('stream_i_X_test_raw.npy')).astype(np.float64)
y_test_raw = np.load(resolve_artifact('stream_i_y_test_raw.npy')).astype(np.float64)
train_meta = pd.read_parquet(resolve_artifact('stream_i_train_metadata.parquet'))
test_meta = pd.read_parquet(resolve_artifact('stream_i_test_metadata.parquet'))
feature_map = json.loads(resolve_artifact('stream_i_feature_map.json').read_text())
FEATURE_NAMES = feature_map['features']

if X_train_raw.shape[1] != len(FEATURE_NAMES):
    raise AssertionError('Feature count does not match feature map.')
if len(X_train_raw) != len(y_train_raw) or len(X_test_raw) != len(y_test_raw):
    raise AssertionError('Feature and target lengths do not match.')
if not np.isfinite(X_train_raw).all() or not np.isfinite(X_test_raw).all():
    raise ValueError('Non-finite input features detected.')

RANDOM_STATE = 34
N_TRIALS = 100
MAX_EPOCHS = 80
CV_SPLITS = 3
CV_TEST_SIZE = 13
OPTUNA_TIMEOUT_MINUTES = 240

## 1. Define the model and conditional search space

In [ ]:
MODEL_KEY = 'sgd_huber'
MODEL_TITLE = '4IB — SGDRegressor (Huber Loss)'
FIXED_LOSS = 'huber'


def suggest_params(trial):
    penalty = trial.suggest_categorical('penalty', ['l2', 'l1', 'elasticnet', None])
    params = {
        'loss': FIXED_LOSS,
        'penalty': penalty,
        'alpha': trial.suggest_float('alpha', 1e-7, 1e-2, log=True),
        'learning_rate': trial.suggest_categorical('learning_rate', ['constant', 'optimal', 'invscaling', 'adaptive']),
        'average': trial.suggest_categorical('average', [False, True]),
        'manual_shuffle': trial.suggest_categorical('manual_shuffle', [False, True]),
        'patience': trial.suggest_categorical('patience', [6, 10, 14]),
        'min_delta': trial.suggest_categorical('min_delta', [0.0, 1e-4, 1e-3]),
    }
    if penalty == 'elasticnet':
        params['l1_ratio'] = trial.suggest_float('l1_ratio', 0.05, 0.95)
    else:
        params['l1_ratio'] = 0.15
    if FIXED_LOSS in ('huber', 'epsilon_insensitive'):
        params['epsilon'] = trial.suggest_float('epsilon', 1e-3, 2.0, log=True)
    else:
        params['epsilon'] = 0.1
    if params['learning_rate'] in ('constant', 'invscaling', 'adaptive'):
        params['eta0'] = trial.suggest_float('eta0', 1e-5, 5e-2, log=True)
    else:
        params['eta0'] = 0.01
    if params['learning_rate'] == 'invscaling':
        params['power_t'] = trial.suggest_float('power_t', 0.05, 0.9)
    else:
        params['power_t'] = 0.25
    return params


def make_estimator(params):
    return SGDRegressor(
        loss=params['loss'], penalty=params['penalty'], alpha=params['alpha'],
        l1_ratio=params.get('l1_ratio', 0.15), fit_intercept=True,
        max_iter=1, tol=None, shuffle=False, verbose=0,
        epsilon=params['epsilon'], random_state=RANDOM_STATE,
        learning_rate=params['learning_rate'], eta0=params['eta0'],
        power_t=params['power_t'], early_stopping=False,
        warm_start=True, average=params['average'],
    )

## 2. Fold-safe scaling and external early stopping

`partial_fit()` performs one pass/update epoch, so early stopping is implemented explicitly on each chronological validation block. Feature scaling and target `log1p`/standardization are fitted on the fold's training subset only. Validation predictions are converted back to accident counts before MAE is calculated.

In [ ]:
def make_target_scaler(y_fit):
    log_y = np.log1p(np.maximum(np.asarray(y_fit, dtype=np.float64), 0.0)).reshape(-1, 1)
    scaler = StandardScaler().fit(log_y)
    return scaler


def inverse_target(y_scaled, y_scaler):
    log_y = y_scaler.inverse_transform(np.asarray(y_scaled).reshape(-1, 1)).ravel()
    return np.maximum(np.expm1(np.clip(log_y, -20.0, 20.0)), 0.0)


def fit_one_fold(params, x_fit_raw, y_fit_raw, x_val_raw, y_val_raw, seed=34):
    x_scaler = StandardScaler().fit(x_fit_raw)
    y_scaler = make_target_scaler(y_fit_raw)
    x_fit = x_scaler.transform(x_fit_raw).astype(np.float64)
    x_val = x_scaler.transform(x_val_raw).astype(np.float64)
    y_fit_scaled = y_scaler.transform(np.log1p(np.maximum(y_fit_raw, 0.0)).reshape(-1, 1)).ravel()

    estimator = make_estimator(params)
    best_model = None
    best_score = float('inf')
    best_epoch = 0
    wait = 0
    rng = np.random.default_rng(seed)
    patience = int(params['patience'])
    min_delta = float(params['min_delta'])

    for epoch in range(1, MAX_EPOCHS + 1):
        order = rng.permutation(len(x_fit)) if params['manual_shuffle'] else np.arange(len(x_fit))
        estimator.partial_fit(x_fit[order], y_fit_scaled[order])
        pred_scaled = estimator.predict(x_val)
        pred_counts = inverse_target(pred_scaled, y_scaler)
        if not np.isfinite(pred_counts).all():
            score = float('inf')
        else:
            score = float(mean_absolute_error(y_val_raw, pred_counts))
        if score < best_score - min_delta:
            best_score = score
            best_epoch = epoch
            best_model = copy.deepcopy(estimator)
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break
    if best_model is None or not np.isfinite(best_score):
        raise optuna.TrialPruned('No finite validation prediction was obtained.')
    return best_model, x_scaler, y_scaler, best_score, best_epoch


def make_temporal_folds(n_samples):
    test_size = min(CV_TEST_SIZE, max(8, n_samples // 20))
    if n_samples <= (CV_SPLITS + 1) * test_size:
        raise ValueError(f'Not enough training samples ({n_samples}) for {CV_SPLITS} expanding folds.')
    splitter = TimeSeriesSplit(n_splits=CV_SPLITS, test_size=test_size)
    return list(splitter.split(np.arange(n_samples)))

FOLDS = make_temporal_folds(len(X_train_raw))
print('Temporal folds:', [(len(tr), len(va)) for tr, va in FOLDS])

## 3. Run or resume the Optuna study

In [ ]:
STUDY_DB = MODEL_DIR / f'{MODEL_KEY}_optuna.sqlite3'
STUDY_NAME = f'us_accidents_weekly_{MODEL_KEY}'
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='minimize',
    sampler=TPESampler(seed=RANDOM_STATE, n_startup_trials=12),
    pruner=MedianPruner(n_startup_trials=12, n_warmup_steps=1),
    storage=f'sqlite:///{STUDY_DB}',
    load_if_exists=True,
)
remaining_trials = max(0, N_TRIALS - len(study.trials))
print('Existing trials:', len(study.trials), '| additional trials:', remaining_trials)


def objective(trial):
    params = suggest_params(trial)
    fold_scores, fold_epochs = [], []
    for fold_id, (train_idx, val_idx) in enumerate(FOLDS):
        try:
            _, _, _, score, best_epoch = fit_one_fold(
                params,
                X_train_raw[train_idx], y_train_raw[train_idx],
                X_train_raw[val_idx], y_train_raw[val_idx],
                seed=RANDOM_STATE,
            )
        except (FloatingPointError, ValueError, OverflowError) as exc:
            raise optuna.TrialPruned(f'Invalid training configuration: {exc}')
        fold_scores.append(score)
        fold_epochs.append(best_epoch)
        trial.report(float(np.mean(fold_scores)), step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()
    trial.set_user_attr('fold_mae', fold_scores)
    trial.set_user_attr('fold_best_epochs', fold_epochs)
    return float(np.mean(fold_scores))

if remaining_trials:
    study.optimize(
        objective, n_trials=remaining_trials, timeout=OPTUNA_TIMEOUT_MINUTES * 60,
        n_jobs=1, gc_after_trial=True, show_progress_bar=True,
    )
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
if not completed:
    raise RuntimeError('Optuna produced no completed trials. Inspect the failed/pruned trial table.')
print('Best CV MAE:', study.best_value)
print('Best parameters:', study.best_params)

## 4. Refit on the complete training period and evaluate the frozen model

In [ ]:
best_trial = study.best_trial
best_params = suggest_params_from_best = dict(best_trial.params)
# Reconstruct conditional parameters in the exact estimator schema.
if MODEL_KEY.startswith('sgd_'):
    best_params['loss'] = FIXED_LOSS
    best_params.setdefault('alpha', 1e-4)
    best_params['manual_shuffle'] = bool(best_params.get('manual_shuffle', False))
    best_params['min_delta'] = float(best_params.get('min_delta', 0.0))
    best_params['patience'] = int(best_params.get('patience', 10))
    if best_params.get('penalty') != 'elasticnet': best_params['l1_ratio'] = 0.15
    if best_params.get('learning_rate') != 'invscaling': best_params['power_t'] = 0.25
    if best_params.get('learning_rate') == 'optimal': best_params['eta0'] = 0.01
    if FIXED_LOSS == 'squared_error': best_params['epsilon'] = 0.1
else:
    best_params['hidden_layer_sizes'] = HIDDEN_ARCHITECTURES[best_params.pop('hidden_architecture')]
    best_params['manual_shuffle'] = bool(best_params.get('manual_shuffle', False))
    best_params['min_delta'] = float(best_params.get('min_delta', 0.0))
    best_params['patience'] = int(best_params.get('patience', 10))
    if best_params['solver'] == 'adam':
        best_params.update({'learning_rate':'constant','momentum':0.9,'nesterovs_momentum':True,'power_t':0.5})
    else:
        best_params.update({'beta_1':0.9,'beta_2':0.999})

# Fit scalers only on all training observations, then run the selected number of partial_fit epochs.
final_x_scaler = StandardScaler().fit(X_train_raw)
final_y_scaler = make_target_scaler(y_train_raw)
X_fit = final_x_scaler.transform(X_train_raw).astype(np.float64)
y_fit = final_y_scaler.transform(np.log1p(np.maximum(y_train_raw, 0.0)).reshape(-1, 1)).ravel()
final_model = make_estimator(best_params)
fold_epochs = best_trial.user_attrs.get('fold_best_epochs', [20])
final_epochs = int(np.clip(np.median(fold_epochs), 1, MAX_EPOCHS))
rng = np.random.default_rng(RANDOM_STATE)
for epoch in range(final_epochs):
    order = rng.permutation(len(X_fit)) if best_params['manual_shuffle'] else np.arange(len(X_fit))
    final_model.partial_fit(X_fit[order], y_fit[order])

X_test_scaled = final_x_scaler.transform(X_test_raw).astype(np.float64)
test_pred_scaled = final_model.predict(X_test_scaled)
test_predictions = inverse_target(test_pred_scaled, final_y_scaler)

prediction_frame = test_meta.copy()
prediction_frame['actual_count'] = y_test_raw.astype(float)
prediction_frame['predicted_count'] = test_predictions.astype(float)
prediction_frame['model'] = MODEL_TITLE
prediction_path = REPORT_DIR / f'{MODEL_KEY}_test_predictions.csv'
prediction_frame.to_csv(prediction_path, index=False)

bundle = {
    'model': final_model,
    'x_scaler': final_x_scaler,
    'y_scaler': final_y_scaler,
    'feature_names': FEATURE_NAMES,
    'model_key': MODEL_KEY,
    'model_title': MODEL_TITLE,
    'best_params': best_params,
    'final_epochs': final_epochs,
    'random_state': RANDOM_STATE,
    'forecast_horizon': 1,
    'week_definition': feature_map['split']['week_definition'],
}
model_path = MODEL_DIR / f'{MODEL_KEY}_bundle.joblib'
joblib.dump(bundle, model_path)

study.trials_dataframe().to_csv(REPORT_DIR / f'{MODEL_KEY}_optuna_trials.csv', index=False)
metadata = {
    'model_key': MODEL_KEY, 'model_title': MODEL_TITLE,
    'best_cv_mae': float(study.best_value), 'best_trial_number': int(best_trial.number),
    'best_params': {k: (list(v) if isinstance(v, tuple) else v) for k, v in best_params.items()},
    'final_epochs': final_epochs, 'n_trials_total': len(study.trials),
    'completed_trials': len(completed), 'cv_splits': CV_SPLITS,
    'max_epochs_per_fold': MAX_EPOCHS, 'random_state': RANDOM_STATE,
    'compute': 'CPU only', 'prediction_file': str(prediction_path), 'model_file': str(model_path),
}
(REPORT_DIR / f'{MODEL_KEY}_metadata.json').write_text(json.dumps(metadata, indent=2, default=str))

print('Saved model:', model_path)
print('Saved predictions:', prediction_path)
print('Final fit epochs:', final_epochs)

## 5. Quick prediction sanity check

In [ ]:
print(prediction_frame[['target_week_id','target_week_start','actual_count','predicted_count']].head(10))
print('Predictions finite:', np.isfinite(test_predictions).all())
print('Negative raw predictions after inverse transform:', int((test_predictions < 0).sum()))
print('Test MAE (descriptive; final metrics are consolidated in Notebook 6):',
      float(mean_absolute_error(y_test_raw, test_predictions)))